<a href="https://colab.research.google.com/github/snehakokatanur5-sys/1MS25IS117_assignment1/blob/main/1MS25IS117_assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [ ]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [ ]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-dataproc 5.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-clo

In [3]:

import os
from google.colab import userdata

# Use the exact Secret name shown in your Colab Secrets panel.
# If your Secret is named GOOGLE_API_KEY:
api_key = userdata.get("GOOGLE_API_KEY")

# Remove accidental spaces or newlines
api_key = "".join(api_key.split())

# Make the key available to the Gemini SDK
os.environ["GOOGLE_API_KEY"] = api_key

print("API key loaded successfully:", bool(api_key))

API key loaded successfully: True


In [4]:
# @title User input needed.

# @title User input needed.

# Add four documents for the search agent to use.
DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section. "
        "Documents can be related to any topic, such as sports, manga, food, or music. "
        "Each document must contain at least four sentences. "
        "Use a minimum of four documents to test the search agent."
    ),
    "fill_in1": (
        "Artificial Intelligence is a branch of computer science that develops systems capable of performing tasks "
        "that normally require human intelligence. These tasks include understanding language, recognizing images, "
        "solving problems, and making predictions. Machine learning is an important part of artificial intelligence "
        "because it allows computers to learn patterns from data. AI is used in healthcare, education, finance, and transportation."
    ),
    "fill_in2": (
        "Machine learning enables computers to learn from examples instead of relying only on explicitly programmed rules. "
        "Supervised learning uses labeled data to train a model. Unsupervised learning identifies patterns in data without "
        "predefined labels. Reinforcement learning trains an agent through rewards and penalties based on its actions."
    ),
    "fill_in3": (
        "Generative AI can create new content, including text, images, music, and computer code. "
        "Large language models are trained on large collections of text to generate responses to prompts. "
        "These models can assist with writing, summarization, translation, and programming. "
        "Their outputs should still be checked for accuracy because they can sometimes generate incorrect information."
    ),
    "fill_in4": (
        "AI agents can use tools to gather information and complete tasks. A search tool allows an agent to retrieve "
        "relevant information from a collection of documents. A ReAct agent combines reasoning with actions and observations "
        "to decide what to do next. The agent can use search results to improve its response instead of relying only on its initial knowledge."
    )
}

In [5]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))


[Source: sample_protocol]
Students must fill in documents in this section. Documents can be related to any topic, such as sports, manga, food, or music. Each document must contain at least four sentences. Use a minimum of four documents to test the search agent.

[Source: fill_in4]
AI agents can use tools to gather information and complete tasks. A search tool allows an agent to retrieve relevant information from a collection of documents. A ReAct agent combines reasoning with actions and observations to decide what to do next. The agent can use search results to improve its response instead of relying only on its initial knowledge.


In [10]:
# @title Ask gemini to search for something
# import google.generativeai as genai

from google import genai
from google.genai import types
import os

# Initialize Gemini client
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# Configure the search tool
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the document collection for relevant information.",
            parameters={
                "type": "OBJECT",
                "properties": {
                    "query": {
                        "type": "STRING",
                        "description": "Search query"
                    }
                },
                "required": ["query"]
            }
        )
    ]
)

# Configure the model
config = types.GenerateContentConfig(
    tools=[search_tool],
    system_instruction="""
    You are a chef training at a Michelin star restaurant.

    When you need specific information about temperatures,
    ingredients, cooking methods, or cooking time, use the
    search_documents tool.

    Only answer based on information found in the documents.
    If the documents do not contain the answer, say so clearly.
    """,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    )
)

MODEL_NAME = "gemini-3.8-flash"

print("Model ready:", MODEL_NAME)


Model ready: gemini-3.8-flash


In [14]:



import os
import time
from google.colab import userdata
from google import genai
from google.genai import types

# --------------------------------------------------
# 1. Load API Key from Google Colab Secrets
# --------------------------------------------------

api_key = userdata.get("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY not found. Check your Colab Secrets."
    )

os.environ["GOOGLE_API_KEY"] = api_key

print("API key loaded successfully.")

# --------------------------------------------------
# 2. Initialize Gemini Client
# --------------------------------------------------

client = genai.Client(api_key=api_key)

# MODEL_NAME = "gemini-3.8-flash"
MODEL_NAME = "gemini-3.5-flash-lite"

print("Gemini client initialized:", MODEL_NAME)

# --------------------------------------------------
# 3. Define Search Tool
# --------------------------------------------------

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description=(
                "Search the document collection to find "
                "the most relevant information."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "query": {
                        "type": "STRING",
                        "description": "Search query"
                    }
                },
                "required": ["query"]
            }
        )
    ]
)

# --------------------------------------------------
# 4. Configure Gemini Model
# --------------------------------------------------

config = types.GenerateContentConfig(
    tools=[search_tool],
    system_instruction="""
    You are a chef training at a Michelin star restaurant.

    When you need specific information about temperatures,
    ingredients, cooking methods, or cooking time, use the
    search_documents tool.

    Only answer based on information found in the documents.

    If the documents do not contain the answer,
    say so clearly.
    """,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    )
)

# --------------------------------------------------
# 5. Send Request with Retry Handling
# --------------------------------------------------

def send_with_retry(contents, max_retries=4):

    for attempt in range(max_retries):

        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config
            )

            return response

        except Exception as e:

            error = str(e)
            error_lower = error.lower()

            # Daily quota errors cannot be fixed by short retries
            if "429" in error and (
                "per day" in error_lower
                or "quota" in error_lower
            ):
                print(
                    "Gemini API daily quota exceeded. "
                    "Please try again after the quota resets."
                )
                raise

            # Retry temporary rate limits and server errors
            if (
                "429" in error
                or "503" in error
                or "UNAVAILABLE" in error
            ) and attempt < max_retries - 1:

                wait = 30 * (attempt + 1)

                print(
                    f"Temporary Gemini API error: {error[:150]}"
                )
                print(
                    f"Retrying in {wait} seconds... "
                    f"Attempt {attempt + 1}/{max_retries}"
                )

                time.sleep(wait)

            else:
                raise

# --------------------------------------------------
# 6. ReAct Agent Implementation
# --------------------------------------------------

def run_agent(question: str, max_steps: int = 8) -> str:

    print(f"\nQuestion: {question}")
    print("-" * 60)

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part(text=question)
            ]
        )
    ]

    for step in range(1, max_steps + 1):

        response = send_with_retry(contents)

        model_content = response.candidates[0].content

        # Preserve the complete model response
        contents.append(model_content)

        # Identify requested tool calls
        calls = [
            part.function_call
            for part in model_content.parts
            if part.function_call
        ]

        # No tool call means final response
        if not calls:

            print("\nFinal answer:")
            print(response.text)

            return response.text

        # Execute tool calls
        result_parts = []

        for call in calls:

            args = dict(call.args)

            print(
                f"\nStep {step}: "
                f"Model calls {call.name}({args})"
            )

            if call.name == "search_documents":

                result = search_documents(**args)

            else:

                result = f"Unknown tool: {call.name}"

            print(
                f"Result preview: {str(result)[:200]}..."
            )

            # Send tool result back to Gemini
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={
                        "result": result
                    }
                )
            )

        contents.append(
            types.Content(
                role="user",
                parts=result_parts
            )
        )

    print("Maximum agent steps reached.")

    return "(Stopped: hit max_steps)"

# --------------------------------------------------
# 7. Test the ReAct Agent
# --------------------------------------------------

_ = run_agent(
    "What topics are students supposed to share documents on?"
)



API key loaded successfully.
Gemini client initialized: gemini-3.5-flash-lite

Question: What topics are students supposed to share documents on?
------------------------------------------------------------

Step 1: Model calls search_documents({'query': 'topics students share documents on'})
Result preview: [Source: sample_protocol]
Students must fill in documents in this section. Documents can be related to any topic, such as sports, manga, food, or music. Each document must contain at least four senten...

Final answer:
Based on the documents, students can share documents related to any topic, such as sports, manga, food, or music. Each document must contain at least four sentences.


In [15]:
# @title Ask follow up questions


questions = [
    "What topics are students supposed to share documents on?",
    "What is artificial intelligence, and what are its applications?",
    "What are the different types of machine learning?",
    "What is generative AI, and what are large language models?",
    "What is a ReAct agent, and how does it use a search tool?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # Stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: What topics are students supposed to share documents on?
------------------------------------------------------------

Step 1: Model calls search_documents({'query': 'students share documents topics'})
Result preview: [Source: sample_protocol]
Students must fill in documents in this section. Documents can be related to any topic, such as sports, manga, food, or music. Each document must contain at least four senten...

Final answer:
Based on the documents, students are supposed to share documents on topics such as sports, manga, food, or music.


Question: What is artificial intelligence, and what are its applications?
------------------------------------------------------------

Step 1: Model calls search_documents({'query': 'artificial intelligence applications'})
Result preview: [Source: fill_in1]
Artificial Intelligence is a branch of computer science that develops systems capable of performing tasks that normally require human intelligence. These tasks include understa

In [16]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)